This file is used to add noise to latent. 

The add-noise method is Dynamic Adaptive Noise Addition (DANA).

Based on whether a video is classified as high-motion or low-motion, it
constructs a custom noise by blending frame-consistent "static noise" with
frame-unique "diverse noise". This blended noise is then added to the clean
latents using a standard forward diffusion process. The resulting noisy latents
are saved to a file, ready to be used as the starting point for a video
diffusion model.

## CONFIGURATIONS

In [ ]:
CONFIG = {
    "latent_path": "data/metadata/videos_latents.pt",
    "optical_flow_path": "data/metadata/optical_flow_score.npy",
    "save_path": "data/metadata/noise_videos_latents.pt",
    
    "time_steps": 500,
    "seed": 42,
    "optical_flow_threshold": 1.799,  # 用于分类运动强度高低的阈值
    "high_motion_beta": 0.3,  # 高运动视频的静态噪声比例
    "low_motion_beta": 0.2,   # 低运动视频的静态噪声比例
}

## Set seeds

In [ ]:
import os
import random
import numpy as np
import torch

def set_seed(seed):
    """设置随机种子以确保所有库的可重现性。

    Args:
        seed (int): 用于所有随机数生成器的种子。
    """
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

# 设置随机种子
set_seed(CONFIG["seed"])

## Load data

In [24]:
def load_and_prepare_data(
        latents_path: str,
        optical_flow_path: str) -> tuple[torch.Tensor, np.ndarray]:
    """
    Loads latent vectors and processes optical flow scores into motion labels.

    Args:
        latents_path (str): Path to the .npy file with clean latent vectors.
        optical_flow_path (str): Path to the .npy file with optical flow scores.

    Returns:
        Tuple[torch.Tensor, np.ndarray]: A tuple containing:
            - The loaded latent vectors as a PyTorch tensor.
            - The processed binary motion labels as a NumPy array.
    """
    clean_latents = torch.load(latents_path) # (250, 12, 4, 36, 64)
    optical_flow_scores = np.load(optical_flow_path) # (250)

    motion_labels = np.where(optical_flow_scores >= CONFIG["optical_flow_threshold"],1, 0) # (250)
    
    return clean_latents, motion_labels

## Define add-noise policy

In [25]:
import math
import torch.nn.functional as F

class Diffusion(object):
    """
    Handles the forward diffusion process (adding noise to data).

    This class sets up the noise schedule and provides a method to apply
    a custom blended noise to an initial latent vector `x_0`.
    """

    def __init__(self, time_steps: int):
        """Initializes the diffusion schedule parameters.

        Args:
            time_steps (int): The total number of steps in the diffusion process.
        """
        self.time_steps = time_steps
        self.betas = self._linear_beta_schedule()

        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, axis=0)
        self.alphas_cumprod_prev = F.pad(self.alphas_cumprod[:-1], (1, 0), value=1.0)
        self.sqrt_recip_alphas = torch.sqrt(1.0 / self.alphas)
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod)
        self.posterior_variance = self.betas * (1.0 - self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def _get_index_from_list(self, vals: torch.Tensor, t: torch.Tensor,
                             x_shape: torch.Size) -> torch.Tensor:
        """Extracts values from a 1D tensor using indices from `t`.

        Args:
            vals (torch.Tensor): The 1D tensor of values (e.g., alphas).
            t (torch.Tensor): A tensor of indices.
            x_shape (torch.Size): The shape of the target tensor `x` to which
              the values will be broadcast.

        Returns:
            torch.Tensor: The selected values, reshaped to be broadcastable
              to `x_shape`.
        """
        batch_size = t.shape[0]
        out = vals.gather(-1, t.cpu())
        return out.reshape(batch_size, *((1, ) * (len(x_shape) - 1))).to(t.device)

    def _linear_beta_schedule(self,
                              start: float = 0.0001,
                              end: float = 0.02) -> torch.Tensor:
        """Generates a linear noise schedule.

        Args:
            start (float): The starting value for beta.
            end (float): The ending value for beta.

        Returns:
            torch.Tensor: A 1D tensor of beta values.
        """
        return torch.linspace(start, end, self.time_steps)

    def forward(self, x_0: torch.Tensor,
                dynamic_beta: float) -> torch.Tensor:
        """
        Applies dynamic noise to the input tensor for a random timestep.

        This is the core of the DANA process. It creates a blended noise from
        static and diverse components, scaled by `dynamic_beta`, and applies
        it to the clean latent `x_0`.

        Args:
            x_0 (torch.Tensor): The clean input tensor (latents) of shape
              (b, f, c, h, w).
            dynamic_beta (float): The blending factor for static vs. diverse
              noise. Higher values give more weight to static noise.

        Returns:
            torch.Tensor: The noised tensor `x_t`.
        """
        b, f, c, h, w = x_0.shape # (250, 12, 4, 36, 64)
        t = torch.randint(0, self.time_steps, (b, )).to(self.device).long() # (250,)

        # Generate diverse noise (unique per frame) and static noise (same across frames)
        diverse_noise = torch.randn_like(x_0).to(self.device)
        same_noise_i = torch.randn((b, 1, c, h, w)).to(self.device)
        same_noise = same_noise_i.repeat(1, f, 1, 1, 1)

        # Blend the two noise types using the dynamic_beta
        # Note: The sum of variances is (1-beta) + beta = 1, so the total noise has unit variance.
        diverse_noise = diverse_noise * math.sqrt(1 - dynamic_beta)
        same_noise = same_noise * math.sqrt(dynamic_beta)
        total_noise = diverse_noise + same_noise

        # Get the appropriate alpha values for the random timestep t
        sqrt_alphas_cumprod_t = self._get_index_from_list(
            self.sqrt_alphas_cumprod, t, x_0.shape)
        sqrt_one_minus_alphas_cumprod_t = self._get_index_from_list(
            self.sqrt_one_minus_alphas_cumprod, t, x_0.shape)

        # Apply noise using the standard diffusion formula: x_t = sqrt(alpha_t) * x_0 + sqrt(1 - alpha_t) * noise
        return (sqrt_alphas_cumprod_t.to(self.device) * x_0.to(self.device) +
                sqrt_one_minus_alphas_cumprod_t.to(self.device) *
                total_noise.to(self.device))

## Add noise to latent

In [26]:
from tqdm.auto import tqdm

def apply_dynamic_noise(clean_latents: torch.Tensor,
                        motion_labels: np.ndarray,
                        diffusion_model: Diffusion) -> torch.Tensor:
    """
    Iterates through latents and applies DANA based on motion labels.

    Args:
        clean_latents (torch.Tensor): Tensor of clean latent vectors.
        motion_labels (np.ndarray): Array of binary motion labels.
        diffusion_model (Diffusion): An instantiated Diffusion object.

    Returns:
        torch.Tensor: A tensor containing all the noised latent vectors.
    """
    noisy_latents_list = []
    for i in tqdm(range(len(clean_latents)), desc="Applying Dynamic Noise"):
        # Select the dynamic beta based on the motion label for the current video.
        is_high_motion = (motion_labels[i] == 1)
        dynamic_beta = CONFIG["high_motion_beta"] if is_high_motion else CONFIG["low_motion_beta"]

        # Apply the forward diffusion process with the selected dynamic beta.
        # Add a batch dimension to the single latent vector.
        noisy_latent = diffusion_model.forward(clean_latents[i:i + 1],
                                               dynamic_beta)
        noisy_latents_list.append(noisy_latent)

    return torch.cat(noisy_latents_list, dim=0)

## Define main function

In [ ]:

def main():
    """
    DANA脚本的主执行函数。
    """
    print("🚀 开始动态自适应噪声添加 (DANA) 流程...")
    print(f"配置参数: 时间步数={CONFIG['time_steps']}, 光流阈值={CONFIG['optical_flow_threshold']}")
    
    print("\n📂 加载和准备数据...")
    clean_latents, motion_labels = load_and_prepare_data(CONFIG["latent_path"], CONFIG["optical_flow_path"])
    print(f"✅ 已加载 {clean_latents.shape[0]} 个潜在向量，形状: {clean_latents.shape}")
    print(f"✅ 已处理 {motion_labels.shape[0]} 个运动标签")
    
    # 统计高/低运动视频数量
    high_motion_count = np.sum(motion_labels == 1)
    low_motion_count = np.sum(motion_labels == 0)
    print(f"📊 高运动视频: {high_motion_count} 个, 低运动视频: {low_motion_count} 个")

    # 实例化扩散模型
    print(f"\n🔧 初始化扩散模型 (时间步数: {CONFIG['time_steps']})...")
    diffusion_model = Diffusion(time_steps=CONFIG["time_steps"])

    print("\n🎯 应用动态噪声...")
    noisy_latents = apply_dynamic_noise(clean_latents, motion_labels, diffusion_model)

    print(f"\n💾 保存噪声潜在向量，形状: {noisy_latents.shape}...")
    torch.save(noisy_latents, CONFIG["save_path"])
    print(f"✅ 完成！噪声潜在向量已保存到: {CONFIG['save_path']}")


## Start!

In [28]:

if __name__ == '__main__':
    main()

Loading and preparing data...
Loaded 250 latent vectors.
Processed 250 motion labels.


Applying Dynamic Noise:   0%|          | 0/250 [00:00<?, ?it/s]

Applying Dynamic Noise: 100%|██████████| 250/250 [00:01<00:00, 247.50it/s]


Saving noised latents of shape torch.Size([250, 12, 4, 36, 64]) ...
Done.
